### Text corpus preparation and analysis

Part 1:
- Filter unique words from a text using a dictionary
- Remove the punctuation from within and between words
- Compute word frequencies of each unique word
- Validate a text using dictionary subtraction
- Subtract stop-words that carry "no meaning"

#### Getting the data

In [4]:
from pyodide.http import open_url
import os

text = open_url("https://tinyurl.com/brown-txt").read()

with open("brown.txt", 'w') as f:
    f.write(text)                  # write text to brown.txt

for name in os.listdir('.'): # loop over files in the current working directory
    if name == "brown.txt":
        size = os.path.getsize(name)
        print(f"{name}: {size} bytes")

brown.txt: 477688 bytes


#### Create a dictionary of unique words

In [5]:
filename = "brown.txt"
filename

'brown.txt'

In [7]:
unique_words = {}
unique_words
type(unique_words)

dict

In [8]:
f = open(filename)
print(repr(next(f)))
print(repr(next(f)))
print(repr(next(f)))
f.close()
print(f.closed)

'The Project Gutenberg eBook of The innocence of Father Brown\n'
'    \n'
'This eBook is for the use of anyone anywhere in the United States and\n'
True


In [10]:
for line in open(filename):            # every line of the file
    seq = line.split()                 # split line into words
    for word in seq:                   # every word in the line
        unique_words[word] = 1         # every unique word->key, value = 1

len(unique_words)

14307

#### Check the words for validity

In [11]:
trailing_bang = []           # make empty list
for word in unique_words:    # go through every word of the dict
    if word.endswith('!'):   
        trailing_bang.append(word) # add words ending in ! to the list

trailing_bang[:5]

['God!', 'proof!', 'course!', 'like!', 'classes!']

In [12]:
# same code as before as a "list comprehension"
trailing_bang2 = [w for w in unique_words if w.endswith('!')]
trailing_bang2[:5]

['God!', 'proof!', 'course!', 'like!', 'classes!']

In [14]:
[print(w) for w in sorted(unique_words, key=len)[-10:-4]];

Galloways--especially
(trademark/copyright)
inspiration--important
darkness--hieroglyphics
magnificence--something
excrescences--yourselves,


In [ ]:
[print(m) for m in dir(str) if not m.startswith('_')];

#### Getting rid of punctuation

Three string functions are going to be helpful:
- removing hyphens in a line and `replace` them with spaces
- `split` the line to separate the words in the corpus
- use `strip` after splitting to remove punctuation at start and end of words

##### Replace hyphens with whitespace

In [16]:
def split_line(string):
    return string.replace('--',' ').split() # chain command

In [18]:
s = "darkness--hieroglyphics"
result = split_line(s)
print(result)
print(type(result))

['darkness', 'hieroglyphics']
<class 'list'>


##### Remove punctutation 

In [24]:
import unicodedata

uni = unicodedata.category('A')
print(uni)  # Lu = Letter  + uppercase

uni = unicodedata.category('a')
print(uni)  # Lu = Letter  + lowercase

uni = unicodedata.category(':')
print(uni)  # Po = Punctuation  + other

uni = unicodedata.category('^')
print(uni)  # Sk = non-letterlike modifier symbol

uni = unicodedata.category(' ')
print(uni)  # Zs = Space separator

Lu
Ll
Po
Sk
Zs


In [26]:
# store unique punctuation marks in a dictionary
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

In [ ]:
[print(char) for char in punc_marks];

In [31]:
punctuation = ''.join(punc_marks)
punctuation  # list of unique punctuation marks 

'.,-:[#]/*;()’“”?!‘_—•%'

##### Strip punctuation from the beginning and end of words and make them lower case

In [32]:
def clean_word(word):
    return word.strip(punctuation).lower()

In [34]:
foo = clean_word("Superfragilisticexpialidotious!")
foo

'superfragilisticexpialidotious'

In [35]:
clean_word(s)

'darkness--hieroglyphics'

##### Generate a clean text corpus

Use `split_line` and `clean_word` to create a corpus of unique, lowercase words without any punctuation

In [37]:
corpus = {}
for line in open(filename):
    for word in split_line(line):     # remove -- in the line
        word = clean_word(word)       # remove punctuation, convert to lower case
        corpus[word] = 1              # add the cleaned word as a key to corpus

len(corpus)  # cleaning lost us 40% of our text

8564

In [ ]:
keys = list(corpus)    # store keys in a list
[print(k) for k in keys[:10]];   # top 10 keys
[print(k) for k in keys[-10:]];  # last 10 keys

In [42]:
[print(w) for w in sorted(corpus,key=len)[-10:-5]]

multi-millionaire
unsubstantialness
sub-consciousness
dissipated-looking
lieutenant-colonel


[None, None, None, None, None]

In [ ]:
#### Compute word frequencies

In [44]:
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1  # word never seen before
        else:
            word_counter[word] += 1 # word seen before

In [45]:
def second_element(term):
    return term[1]        # give me the second element back

In [47]:
items = sorted(word_counter.items(),    # contains key + value
               key=second_element,      # jkey = lambda t: t[1]
               reverse=True)
print(len(items))
print(type(items))

8564
<class 'list'>


In [49]:
[print(f"{word:<5}{freq}") for word,freq in items[:5]];

the  5539
and  2476
of   2373
a    2316
he   1552


In [55]:
[print(f"{word:<12}{freq}") for word,freq in items[-5:]];

pg          1
facility    1
includes    1
subscribe   1
newsletter  1


In [53]:
word_counter["brown"]

291